In [ ]:
import os
import requests
import pandas as pd
from ldaca.ldaca import LDaCA
# for download progress bar
from tqdm import tqdm
# for parsing out the filename from content-disposition
import pyrfc6266
# interactive UI widgets
import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual

In [ ]:
BASE_URL="https://data.ldaca.edu.au/api"
API_TOKEN="REPLACEME"

In [ ]:
ldaca = LDaCA(
    url=BASE_URL,
    token=API_TOKEN
)

In [ ]:
# Saves the ro-crate-metadata.json in the data_dir
ldaca.retrieve_collection(
    collection='arcp://name,hdl10.25911~m03c-yz22',
    collection_type='Collection',
    data_dir='data')

In [ ]:
speakers = [e.properties() for e in ldaca.crate.get_entities() if "speaker" in e.get("@id")]

ethnicities = set()
age_groups = set()
genders = set()
for speaker in speakers:
    ethnicities |= set(speaker.get('arcp://name,custom/terms#ethnicity', []))
    if type(speaker.get('arcp://name,custom/terms#ageGroup', [])[0]) == dict:
        as_strings = [a['@id'] for a in speaker.get('arcp://name,custom/terms#ageGroup', [])]
        age_groups |= set(as_strings)
    else:
        age_groups |= set(speaker.get('arcp://name,custom/terms#ageGroup', []))
    genders |= set(speaker.get('gender', []))

In [ ]:
age_group_input = widgets.SelectMultiple(
    options=sorted(age_groups),
    value=[],
    description='Age group:',
    disabled=False,
)
ethnicity_input = widgets.SelectMultiple(
    options=sorted(ethnicities),
    value=[],
    description='Ethnicity:',
    disabled=False,
)
gender_input = widgets.SelectMultiple(
    options=sorted(genders),
    value=[],
    description='Gender:',
    disabled=False,
)

widgets.VBox(
    [
        widgets.Label(value='(Hold ctrl/cmd to select multiple)'), 
        age_group_input,
        ethnicity_input,
        gender_input
    ]
)

In [ ]:
def download_file(file_entity, force = False):
    if not file_entity:
        raise Exception(f"{file_entity} is None")
    if not 'File' in file_entity.get("@type", []):
        raise Exception(f"Tried to download {file_entity.get('@id')}, but it does not have type File")

    # TODO: move magic string
    cwd = os.getcwd()
    folder = os.path.join(cwd, "data")

    head_response = requests.head(
        file_entity.get('@id'),
        allow_redirects=True,
        headers={'Authorization': 'Bearer ' + API_TOKEN}
    )

    filename = pyrfc6266.parse_filename(head_response.headers['content-disposition'])
    print(filename, end=' ')

    filepath = os.path.join(folder, filename)

    if os.path.exists(filepath) and not force:
        # TODO: check its the right size
        expected = int(head_response.headers.get("content-length", 0))
        actual = os.path.getsize(filepath)
        if expected == actual:
            print('already downloaded')
            return filepath
        else:
            print(f'exists but unexpected size, deleting ({expected} != {actual})', end=' ')
            os.remove(filepath)
    elif os.path.exists(filepath) and force:
        # TODO: delete existing
        print('exists but `force=True`, deleting', end=' ')
        os.remove(filepath)
        pass

    print('downloading')
    response = requests.get(
        file_entity.get('@id'),
        stream=True,
        allow_redirects=True,
        headers={'Authorization': 'Bearer ' + API_TOKEN}
    )

    # Sizes in bytes.
    total_size = int(response.headers.get("content-length", 0))
    block_size = 1024

    with tqdm(total=total_size, unit="B", unit_scale=True) as progress_bar:
        with open(filepath, "wb") as file:
            for data in response.iter_content(block_size):
                progress_bar.update(len(data))
                file.write(data)

def get_audio(text_entity):
    parts = text_entity.get("hasPart")
    for part in parts:
        if  'audio/x-wav' in part.get('encodingFormat', []):
            return part
        elif part.id.endswith('.wav'):
            return part

def get_transcript(text_entity):
    parts = text_entity.get("hasPart")
    for part in parts:
        if part.id.endswith('.eaf'):
            return part

In [ ]:
from IPython.display import display
download_audio_button = widgets.Button(description="Download audio")
download_transcripts_button = widgets.Button(description="Download transcripts")
output = widgets.Output()

texts = []

def filter(_):
    global texts
    texts.clear()

    for e in ldaca.crate.get_entities():
        speaker = e.get("ldac:speaker", None)
        if speaker:
            speaker = speaker[0]
            ethnicity = speaker.get('arcp://name,custom/terms#ethnicity', [])
            age_group = speaker.get('arcp://name,custom/terms#ageGroup', [])
            gender = speaker.get('gender', [])
            if not set(age_group).isdisjoint(age_group_input.value) \
                and not set(ethnicity).isdisjoint(ethnicity_input.value) \
                and not set(gender).isdisjoint(gender_input.value):
                texts.append(e)
    with output:
        output.clear_output()
        print(f'Selected {len(texts)} objects')
    


def download_audio_button_clicked(_b):
    with output:
        for text in texts:
            download_file(get_audio(text))

def download_transcripts_button_clicked(_b):
    with output:
        for text in texts:
            download_file(get_transcript(text))

download_audio_button.on_click(download_audio_button_clicked)
download_transcripts_button.on_click(download_transcripts_button_clicked)

refresh_button = widgets.Button(
    description='Refresh',
    disabled=False,
    button_style='', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Click me',
    icon='refresh' # (FontAwesome names without the `fa-` prefix)
)
refresh_button.on_click(filter)


display(widgets.VBox([
    refresh_button,
    download_audio_button,
    download_transcripts_button
]), output)

In [ ]:
from IPython.display import FileLink
from pathlib import Path
from tqdm import tqdm
import zipfile
from shutil import make_archive

archive_name = f"sydney_speaks_subset"
make_archive(archive_name, 'zip', "./data")

# zip_file_path = f"./{archive_name}.zip"
# local_path = "./data"

# with zipfile.ZipFile(zip_file_path, mode="w") as archive:
#     for file_path in tqdm(list(Path(local_path).rglob("*")), desc='Adding to Zip '):
#         archive.write(
#             file_path,
#             arcname=file_path.relative_to(local_path)
#         )

# display(FileLink(f'{archive_name}.zip'))